INSTALL

In [ ]:
!pip install -q pytorch-msssim

IMPORT

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from glob import glob
from tqdm import tqdm
from pytorch_msssim import ssim
import math

FIND DATA (UIEB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_path = "/content/drive/MyDrive/Minor_Project/UIEB"

all_images = glob(os.path.join(base_path, "**/*.jpg"), recursive=True) + \
             glob(os.path.join(base_path, "**/*.png"), recursive=True)

print("Total images found:", len(all_images))
print("Sample paths:", all_images[:5])

Total images found: 1840
Sample paths: ['/content/drive/MyDrive/Minor_Project/UIEB/test_90/16_img_.png', '/content/drive/MyDrive/Minor_Project/UIEB/test_90/62_img_.png', '/content/drive/MyDrive/Minor_Project/UIEB/test_90/60_img_.png', '/content/drive/MyDrive/Minor_Project/UIEB/test_90/2_img_.png', '/content/drive/MyDrive/Minor_Project/UIEB/test_90/842_img_.png']


SPLIT RAW / REF

In [ ]:
raw_images = [img for img in all_images if "raw" in img.lower()]
ref_images = [img for img in all_images if "ref" in img.lower() or "reference" in img.lower()]

print("Raw:", len(raw_images))
print("Reference:", len(ref_images))

# fallback
if len(raw_images) == 0 or len(ref_images) == 0:
    print("Error Using fallback split")
    raw_images = all_images[:len(all_images)//2]
    ref_images = all_images[len(all_images)//2:]

Raw: 800
Reference: 890


TRANSFORM

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])


DATASET

In [ ]:
class UIEBDataset(Dataset):
    def __init__(self, raw_list, ref_list, transform=None):
        self.raw_images = sorted(raw_list)
        self.ref_images = sorted(ref_list)
        self.transform = transform

    def __len__(self):
        return min(len(self.raw_images), len(self.ref_images))

    def __getitem__(self, idx):
        raw = Image.open(self.raw_images[idx]).convert("RGB")
        ref = Image.open(self.ref_images[idx]).convert("RGB")

        if self.transform:
            raw = self.transform(raw)
            ref = self.transform(ref)

        return raw, ref


In [ ]:
dataset = UIEBDataset(raw_images, ref_images, transform)
print("Dataset size:", len(dataset))

dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

Dataset size: 800


In [ ]:
print(f"Length of dataloader: {len(dataloader)}")
print(f"Length of dataset: {len(dataset)}")

Length of dataloader: 100
Length of dataset: 800


PIXMAMBA BLOCK : Enhancement Stage

In [ ]:
class PixelMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.forward_mixer = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

        self.backward_mixer = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, H, W = x.shape

        x_seq = x.view(B, C, -1).permute(0, 2, 1)
        x_norm = self.norm(x_seq)

        fwd = self.forward_mixer(x_norm)

        bwd = torch.flip(x_norm, dims=[1])
        bwd = self.backward_mixer(bwd)
        bwd = torch.flip(bwd, dims=[1])

        combined = fwd + bwd

        gate = self.gate(x_norm)
        out = gate * combined + (1 - gate) * x_seq

        out = out.permute(0, 2, 1).view(B, C, H, W)

        return out


MODEL : Encoder and Decoder

In [ ]:
class PIXMambaNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU()
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU()
        )

        self.pixmamba1 = PixelMambaBlock(64)
        self.pixmamba2 = PixelMambaBlock(64)

        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU()
        )

        self.dec2 = nn.Sequential(
            nn.Conv2d(32, 3, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)

        x3 = self.pixmamba1(x2)
        x4 = self.pixmamba2(x3)

        x5 = self.dec1(x4)
        out = self.dec2(x5)

        return out


DEVICE

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PIXMambaNet().to(device)

LOSS

In [ ]:
l1_loss = nn.L1Loss()

def total_loss(pred, target):
    l1 = l1_loss(pred, target)
    ssim_loss = 1 - ssim(pred, target, data_range=1.0)
    return l1 + 0.5 * ssim_loss

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


TRAINING

In [ ]:
epochs = 50

for epoch in range(epochs):
    model.train()
    total = 0

    for raw, ref in tqdm(dataloader):
        raw, ref = raw.to(device), ref.to(device)

        output = model(raw)
        loss = total_loss(output, ref)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1} Loss: {total/len(dataloader):.4f}")


100%|██████████| 100/100 [06:46<00:00,  4.07s/it]


Epoch 1 Loss: 0.3807


100%|██████████| 100/100 [00:57<00:00,  1.75it/s]


Epoch 2 Loss: 0.2127


100%|██████████| 100/100 [00:55<00:00,  1.81it/s]


...

Epoch 48 Loss: 0.1658


100%|██████████| 100/100 [00:55<00:00,  1.79it/s]


Epoch 49 Loss: 0.1629


100%|██████████| 100/100 [00:56<00:00,  1.77it/s]

Epoch 50 Loss: 0.1625


SAVE MODEL

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Minor_Project/underwater_image_enhancement.pth")


INFERENCE + SAVE IMAGES

In [ ]:
def save_enhanced_images(dataloader, model, device, save_path, dataset):
    model.eval()
    os.makedirs(save_path, exist_ok=True)
    print(f"Saving enhanced images to: {save_path}")

    for i, (raw, _) in tqdm(enumerate(dataloader), total=len(dataloader), desc="Saving Enhanced Images"):
        raw = raw.to(device)

        with torch.no_grad():
            out = model(raw)

        for j in range(out.size(0)):
            idx = i * dataloader.batch_size + j

            if idx < len(dataset.raw_images):
                filename = os.path.basename(dataset.raw_images[idx])
            else:
                filename = f"img_batch_{i}_idx_{j}.png"

            img = transforms.ToPILImage()(out[j].cpu())
            img.save(os.path.join(save_path, filename))
    print(f"Successfully saved {len(dataset)} enhanced images.")



In [ ]:
save_path_main = "/content/drive/MyDrive/Minor_Project/enhanced_images_train"
save_enhanced_images(dataloader, model, device, save_path_main, dataset)

Saving enhanced images to: /content/drive/MyDrive/Minor_Project/enhanced_images


Saving Enhanced Images: 100%|██████████| 100/100 [01:14<00:00,  1.34it/s]

Successfully saved 800 enhanced images.


## VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt

def visualize_enhancement_results(dataloader, model, device, num_samples_to_display=5, title_prefix=""):
    model.eval()
    plt.figure(figsize=(25, 5 * num_samples_to_display))

    for i, (raw, ref) in enumerate(dataloader):
        if i >= num_samples_to_display:
            break

        raw = raw.to(device)
        ref = ref.to(device)

        with torch.no_grad():
            enhanced = model(raw)

        raw_img = transforms.ToPILImage()(raw.squeeze(0).cpu())
        ref_img = transforms.ToPILImage()(ref.squeeze(0).cpu())
        enhanced_img = transforms.ToPILImage()(enhanced.squeeze(0).cpu())

        diff = torch.abs(enhanced - ref).squeeze(0).cpu()
        diff_heatmap = diff.mean(dim=0)

        # Display raw image
        plt.subplot(num_samples_to_display, 4, i * 4 + 1)
        plt.imshow(raw_img)
        plt.title(f"{title_prefix} Input (Raw) {i+1}")
        plt.axis('off')

        # Display enhanced image
        plt.subplot(num_samples_to_display, 4, i * 4 + 2)
        plt.imshow(enhanced_img)
        plt.title(f"{title_prefix} Output (Enhanced) {i+1}")
        plt.axis('off')

        # Display reference image
        plt.subplot(num_samples_to_display, 4, i * 4 + 3)
        plt.imshow(ref_img)
        plt.title(f"{title_prefix} Reference {i+1}")
        plt.axis('off')

        # Display heatmap of differences
        plt.subplot(num_samples_to_display, 4, i * 4 + 4)
        plt.imshow(diff_heatmap, cmap='hot', vmin=0, vmax=1)
        plt.title(f"{title_prefix} Difference (Heatmap) {i+1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Create a DataLoader for visualization, ensuring no shuffling for consistent results
visualization_dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
visualize_enhancement_results(visualization_dataloader, model, device, num_samples_to_display=5, title_prefix="Train")

Output hidden; open in https://colab.research.google.com to view.

PSNR

In [ ]:
def psnr(pred, target):
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return 100
    return 20 * math.log10(1.0 / math.sqrt(mse))


VALUATION

In [ ]:
def evaluate_psnr_ssim(model, dataloader, device):
    model.eval()
    psnr_total = 0
    ssim_total = 0

    for raw, ref in dataloader:
        raw, ref = raw.to(device), ref.to(device)

        with torch.no_grad():
            out = model(raw)

        # Calculate PSNR
        psnr_total += psnr(out, ref)
        # Calculate SSIM
        ssim_total += ssim(out, ref, data_range=1.0).item()

    num_batches = len(dataloader)
    return psnr_total / num_batches, ssim_total / num_batches

In [ ]:
avg_psnr_combined, avg_ssim_combined = evaluate_psnr_ssim(model, dataloader, device)
print(f"Average PSNR (combined function): {avg_psnr_combined:.4f}")
print(f"Average SSIM (combined function): {avg_ssim_combined:.4f}")

Average PSNR (combined function): 18.1626
Average SSIM (combined function): 0.8689


--------------------------------------------- TEST DATASET -------------------------------------------------------------

## INFERENCE ON TEST IMAGES

In [ ]:
raw_images_test = [img for img in all_images if "test_90" in img.lower()]
ref_images_test = [img for img in all_images if "reference_90" in img.lower()]

print("test_Raw:", len(raw_images_test))
print("test_Reference:", len(ref_images_test))

if len(raw_images_test) == 0 or len(ref_images_test) == 0:
    print("Error Using fallback split")
    raw_images_test = all_images[:len(all_images)//2]
    ref_images_test = all_images[len(all_images)//2:]

test_Raw: 90
test_Reference: 90


In [ ]:
# Create a dataset and dataloader for the test images
dataset_test = UIEBDataset(raw_images_test, ref_images_test, transform)
print("Test Dataset size:", len(dataset_test))

dataloader_test = DataLoader(dataset_test, batch_size=9, shuffle=False, num_workers=2)

Test Dataset size: 90


In [ ]:
print(f"Length of dataloader_test: {len(dataloader_test)}")
print(f"Length of dataset_test: {len(dataset_test)}")

Length of dataloader_test: 10
Length of dataset_test: 90


### Perform Inference and Save Enhanced Test Images

In [ ]:
save_path_test = "/content/drive/MyDrive/Minor_Project/enhanced_images_test"
save_enhanced_images(dataloader_test, model, device, save_path_test, dataset_test)

Saving enhanced images to: /content/drive/MyDrive/Minor_Project/enhanced_images_test


Saving Enhanced Images: 100%|██████████| 10/10 [01:25<00:00,  8.58s/it]

Successfully saved 90 enhanced images.


## EVALUATION ON TEST IMAGES

In [ ]:
avg_psnr_test, avg_ssim_test = evaluate_psnr_ssim(model, dataloader_test, device)
print(f"Average PSNR on Test Set: {avg_psnr_test:.4f}")
print(f"Average SSIM on Test Set: {avg_ssim_test:.4f}")

Average PSNR on Test Set: 18.1502
Average SSIM on Test Set: 0.8738


## Visualize Test Input, Enhanced Output, and Reference Images

In [ ]:
visualization_dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)
visualize_enhancement_results(visualization_dataloader_test, model, device, num_samples_to_display=5, title_prefix="Test")

Output hidden; open in https://colab.research.google.com to view.

--------------------------------------------- U45 DATASET -------------------------------------------------------------

In [ ]:
base_path_u45 = "/content/drive/MyDrive/Minor_Project/U45"

image_test_u45 = glob(os.path.join(base_path_u45, "**/*.jpg"), recursive=True) + \
             glob(os.path.join(base_path_u45, "**/*.png"), recursive=True)

print("Total images found:", len(image_test_u45))
print("Sample paths:", image_test_u45[:5])

Total images found: 45
Sample paths: ['/content/drive/MyDrive/Minor_Project/U45/31.png', '/content/drive/MyDrive/Minor_Project/U45/3.png', '/content/drive/MyDrive/Minor_Project/U45/12.png', '/content/drive/MyDrive/Minor_Project/U45/11.png', '/content/drive/MyDrive/Minor_Project/U45/42.png']


In [ ]:
class U45Dataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.raw_images = sorted(image_list)
        self.transform = transform

    def __len__(self):
        return len(self.raw_images)

    def __getitem__(self, idx):
        raw = Image.open(self.raw_images[idx]).convert("RGB")
        if self.transform:
            raw = self.transform(raw)
        return raw, raw # Return raw twice to fit the (raw, _) unpacking in dataloader iteration

In [ ]:
dataset_u45 = U45Dataset(image_test_u45, transform)
print("U45 Dataset size:", len(dataset_u45))

dataloader_u45= DataLoader(dataset_u45, batch_size=9, shuffle=False, num_workers=2)

U45 Dataset size: 45


In [ ]:
save_path_u45 = "/content/drive/MyDrive/Minor_Project/enhanced_images_u45"
save_enhanced_images(dataloader_u45, model, device, save_path_u45, dataset_u45)

Saving enhanced images to: /content/drive/MyDrive/Minor_Project/enhanced_images_u45


Saving Enhanced Images: 100%|██████████| 5/5 [00:20<00:00,  4.04s/it]

Successfully saved 45 enhanced images.


In [ ]:
def visualize_u45_input_output(dataloader, model, device, num_samples_to_display=5):
    model.eval()
    plt.figure(figsize=(10, 5 * num_samples_to_display))

    for i, (raw, _) in enumerate(dataloader):
        if i >= num_samples_to_display:
            break

        raw = raw.to(device)

        with torch.no_grad():
            enhanced = model(raw)

        raw_img = transforms.ToPILImage()(raw.squeeze(0).cpu())
        enhanced_img = transforms.ToPILImage()(enhanced.squeeze(0).cpu())

        # Display raw image
        plt.subplot(num_samples_to_display, 2, i * 2 + 1)
        plt.imshow(raw_img)
        plt.title(f"U45 Input {i+1}")
        plt.axis('off')

        # Display enhanced image
        plt.subplot(num_samples_to_display, 2, i * 2 + 2)
        plt.imshow(enhanced_img)
        plt.title(f"U45 Enhanced Output {i+1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# Create a DataLoader for visualization of U45 images
visualization_dataloader_u45 = DataLoader(dataset_u45, batch_size=1, shuffle=False)

print("Visualizing Input and Enhanced Output for U45 Dataset:")
visualize_u45_input_output(visualization_dataloader_u45, model, device, num_samples_to_display=5)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:

avg_psnr_u45, avg_ssim_u45 = evaluate_psnr_ssim(model, dataloader_u45, device)
print(f"Average PSNR on U45 Set: {avg_psnr_u45:.4f}")
print(f"Average SSIM on U45 Set: {avg_ssim_u45:.4f}")

Average PSNR on U45 Set: 15.7368
Average SSIM on U45 Set: 0.7414
